<!-- COMMONS LAUNCHER v3 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building Synapsa, an AI-native
learning platform.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T03-L02-bpe-merge-loop-in-cpp/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T03-L02-bpe-merge-loop-in-cpp/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T03-L02-bpe-merge-loop-in-cpp/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy and matplotlib, which Colab, Kaggle,
Binder and Codespaces already have, and a C or C++ compiler (`clang` or `gcc`). The cell
below installs `tokenizers==0.23.2` and fetches the files it needs beside it, and does
nothing where they are already present. On Kaggle, switch Internet on in the notebook's
settings first; Kaggle allows that only for phone-verified accounts.

In [ ]:
# --- COMMONS LAUNCHER v3 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = [("tokenizers", "tokenizers==0.23.2")]            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = ["Makefile", "lesson.cpp", "assets/SOURCE.md", "assets/corpus.txt", "assets/make_corpus.py"]    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T03-L02-bpe-merge-loop-in-cpp/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# T03-L02 · The BPE merge loop, in C++

**You will build:** a byte-pair-encoding trainer in C++ whose merge list is *identical* to
the Python trainer's from T03-L01, on the same corpus — and a measurement of how much time
the language change bought on your machine.

**Time:** ~60 minutes · **Runs on:** a laptop CPU, no GPU, no download
· **Prerequisites:** `T03-L01-bpe-from-scratch`

By the end you will be able to:
1. Implement the four-function merge loop in C++ — count, argmax, merge, train — with nothing
   but `std::` containers.
2. Measure the naive rescan's cost in both languages on one identical corpus, and report the
   speedup your own machine produces.
3. Show that your C++ merge list matches Python's exactly, and name the one rule that makes
   that reproducibility possible.
4. Explain why the speedup is a constant factor and not an asymptotic one.
5. Say which language each production tokenizer is really written in, and what that does and
   does not change about the argument.

**You edit `lesson.cpp`, not this file.** This notebook builds your C++, feeds it cases, and
checks its answers against a Python trainer that is given to you, already working.

In [ ]:
# Setup: one cell, everything the lesson needs, with versions printed.
import platform
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path
from typing import Callable

# Work that costs real seconds lives inside `if __name__ == "__main__":` blocks, so the
# autograder can import this file without re-running the whole lesson. In a notebook
# __name__ IS "__main__", so every one of those cells runs normally when you run it.
HERE = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
IS_REFERENCE = HERE.name == "solutions"
LESSON_ROOT = HERE.parent if IS_REFERENCE else HERE
CPP_SRC = HERE / "lesson.cpp"                      # the file you edit
CPP_BIN = LESSON_ROOT / "build" / ("lesson_ref" if IS_REFERENCE else "lesson")
CORPUS = LESSON_ROOT / "assets" / "corpus.txt"     # ships with the lesson; nothing is downloaded

HEADLINE_MERGES = 300      # the size of the race in section 8
ORACLE_BYTES = 40_000      # a slice of the corpus, for the checks that compare merge lists
ORACLE_MERGES = 30

print("python", sys.version.split()[0], "·", platform.machine(), platform.system())
print("corpus", CORPUS.name, CORPUS.stat().st_size, "bytes")
print("source", CPP_SRC.relative_to(LESSON_ROOT))
_paths = subprocess.run(["make", "-C", str(LESSON_ROOT), f"PYTHON={sys.executable}", "paths"],
                        capture_output=True, text=True)
print(_paths.stdout.strip() or _paths.stderr.strip())

# Every check and every demo below runs through `_try`, so pressing Run all before you have
# written a line of C++ reaches the progress board at the foot of the notebook instead of
# stopping at the first TODO.
# The four exercises, in the order the notebook meets them: (label, C++ function, where).
_EXERCISES = [("exercise 1", "count_pairs", "section 4"), ("exercise 2", "best_pair", "section 5"),
              ("exercise 3", "apply_merge", "section 6"), ("exercise 4", "train", "section 7")]
_FUNCTION = {label: function for label, function, _ in _EXERCISES}
_STATUS: dict = {}       # label -> "passed" | "failed" | "not started", for the progress board
_WAITING_ON: dict = {}   # label -> the C++ function whose TODO stopped it


def _unfinished(exc: BaseException) -> str:
    """The C++ function whose TODO fired. The binary reports it as `NOT_IMPLEMENTED <name>`."""
    found = re.search(r"NOT_IMPLEMENTED (\w+)", str(exc))
    return found.group(1) if found else ""


def _named(labels: list) -> str:
    """["exercise 4"] -> "exercise 4 (`train`)"; several -> "exercises 1 and 4"."""
    if len(labels) == 1:
        return f"{labels[0]} (`{_FUNCTION[labels[0]]}`)"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


def _try(label: str, check: Callable[[], None], needs: tuple = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so, by name: the binary reports which
    function's TODO it hit. A wrong answer prints the check's own message — which names the
    likely mistake — and the notebook carries on, so one broken exercise never hides the
    feedback on the others. A demo names the exercises it `needs`: until each has passed its
    check, the demo says which one it is waiting for and skips, so it never runs on an answer
    a check has just rejected. Nothing is swallowed: every outcome is recorded for the
    progress board, and a script run exits non-zero there if any check came back wrong.
    """
    waiting = [name for name in needs if _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        check()
    except NotImplementedError as exc:
        stub = _unfinished(exc)
        _STATUS[label], _WAITING_ON[label] = "not started", stub
        owner = [name for name, function in _FUNCTION.items() if function == stub]
        if owner and owner[0] != label:
            print(f"{label}: skipped — needs {_named(owner)} first; it still has its TODO in "
                  "lesson.cpp.")
        else:
            todo = f"`{stub}` in lesson.cpp" if stub else "an exercise in lesson.cpp"
            print(f"{label}: not implemented yet — {todo} still has its TODO. Fill it in, then "
                  "re-run this cell.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"

## 1. The algorithm, in one paragraph and one cell

BPE starts from bytes and invents a new symbol for the commonest adjacent pair, over and
over. Sennrich, Haddow and Birch brought it to neural translation by borrowing a compression
algorithm (sourced in `claims.yaml`); every modern tokenizer still runs this loop.

The Python below is the T03-L01 trainer, reproduced here so this lesson stands alone and so
you have something exact to test against. **Read it — it is the specification your C++ has to
match.** Four steps: pre-tokenise, count, pick, merge.

In [ ]:
BOUNDARY = 256          # the symbol between two words. Never merged across.
FIRST_MERGE_ID = 257    # the k-th merge is given id 257 + k
MIN_COUNT = 2           # a pair that occurs once buys no compression
_WHITESPACE = frozenset(b" \t\n\r\f\v")


def pretokenise(blob: bytes) -> list:
    """Bytes of a word, then BOUNDARY, then the next word. Whitespace itself is dropped."""
    seq, in_word = [], False
    for byte in blob:
        if byte in _WHITESPACE:
            if in_word:
                seq.append(BOUNDARY)
                in_word = False
        else:
            seq.append(byte)
            in_word = True
    if in_word:
        seq.append(BOUNDARY)
    return seq


def count_pairs_py(seq: list) -> dict:
    """Count adjacent pairs, skipping any pair that touches BOUNDARY. One O(n) scan."""
    counts = {}
    prev = seq[0] if seq else BOUNDARY
    for i in range(1, len(seq)):
        cur = seq[i]
        if prev != BOUNDARY and cur != BOUNDARY:
            key = (prev, cur)
            counts[key] = counts.get(key, 0) + 1
        prev = cur
    return counts


def best_pair_py(counts: dict):
    """Highest count wins; ties break towards the smaller (a, b). None for an empty dict."""
    best, best_n = None, 0
    for pair, n in counts.items():
        if best is None or n > best_n or (n == best_n and pair < best):
            best, best_n = pair, n
    return (best[0], best[1], best_n) if best is not None else None


def apply_merge_py(seq: list, a: int, b: int, new_id: int) -> list:
    """Replace every non-overlapping left-to-right occurrence of (a, b) with new_id."""
    out, i, n = [], 0, len(seq)
    while i < n:
        if i + 1 < n and seq[i] == a and seq[i + 1] == b:
            out.append(new_id)
            i += 2
        else:
            out.append(seq[i])
            i += 1
    return out


def train_py(seq: list, n_merges: int) -> dict:
    """Run the naive loop. Returns the merges, the final length and the measured seconds."""
    merges = []
    start = time.perf_counter()
    for k in range(n_merges):
        counts = count_pairs_py(seq)
        winner = best_pair_py(counts)
        if winner is None or winner[2] < MIN_COUNT:
            break
        a, b, count = winner
        new_id = FIRST_MERGE_ID + k
        merges.append((a, b, new_id, count))
        seq = apply_merge_py(seq, a, b, new_id)
    return {"merges": merges, "final_symbols": len(seq), "seconds": time.perf_counter() - start}


def corpus_symbols(path=None, limit: int = 0) -> list:
    """Pre-tokenised corpus, optionally only the first `limit` bytes (cut at whitespace)."""
    blob = (path or CORPUS).read_bytes()
    if limit and limit < len(blob):
        cut = limit
        while cut > 0 and not blob[cut - 1:cut].isspace():
            cut -= 1
        blob = blob[:cut]
    return pretokenise(blob)

In [ ]:
if __name__ == "__main__":
    _demo = train_py(pretokenise(b"the cat sat on the mat the cat sat"), 3)
    print("first merges on a nine-word corpus:")
    for _a, _b, _new, _n in _demo["merges"]:
        _left = chr(_a) if _a < 256 else f"<{_a}>"
        _right = chr(_b) if _b < 256 else f"<{_b}>"
        print(f"  ({_left!r}, {_right!r}) -> {_new}   seen {_n} times")

## 2. The cost model: every merge rescans everything

Look again at `train_py`. Each iteration counts pairs over the *whole* sequence, then
rebuilds the whole sequence. Nothing is remembered between merges. The work is
`n_merges × n_symbols`, and since one merge changes only a handful of positions, almost all
of that scanning re-derives counts that did not change.

Do not take that on faith. Time the Python trainer on three slices of the corpus with the
merge count held fixed, and watch the seconds track the symbol count.

In [ ]:
def python_scaling(fractions=(0.25, 0.5, 1.0), merges: int = 30) -> list:
    """Time train_py on increasing slices. Rows of measured numbers, nothing typed by hand."""
    total = CORPUS.stat().st_size
    rows = []
    for fraction in fractions:
        seq = corpus_symbols(limit=int(total * fraction))
        result = train_py(list(seq), merges)
        rows.append({
            "bytes": int(total * fraction),
            "symbols": len(seq),
            "seconds": result["seconds"],
            "ns_per_symbol_per_merge": 1e9 * result["seconds"] / (len(seq) * merges),
        })
    return rows


if __name__ == "__main__":
    print(f"{'bytes':>8} {'symbols':>9} {'time':>9} {'per symbol, per merge':>22}")
    for _row in python_scaling():
        print(f"{_row['bytes']:8d} {_row['symbols']:9d} {_row['seconds']:7.3f} s"
              f" {_row['ns_per_symbol_per_merge']:19.1f} ns")
    print("\nSeconds roughly double when the corpus doubles. The last column is flat, and it")
    print("is the constant you are about to attack: the cost of one symbol, one time, here.")

## 3. Your C++ binary, and how this notebook talks to it

`lesson.cpp` compiles to a small command-line tool. The notebook builds it with `make` and
calls one sub-command per exercise, so each function is graded on its own: an `apply_merge`
that works still scores even while `train` is a TODO.

Run this cell before writing any C++. It should fail — loudly, and by name.

In [ ]:
_BUILD_LOG = {}


def build_cpp(force: bool = False) -> Path:
    """Compile CPP_SRC to CPP_BIN via the lesson Makefile. Cached: it builds once per session."""
    if _BUILD_LOG.get("built") and not force:
        return CPP_BIN
    if shutil.which("make") is None:
        raise RuntimeError("no `make` on PATH — this lesson needs make and a C++17 compiler")
    proc = subprocess.run(
        ["make", "-C", str(LESSON_ROOT),
         f"SRC={CPP_SRC.relative_to(LESSON_ROOT)}",
         f"BIN={CPP_BIN.relative_to(LESSON_ROOT)}", "all"],
        capture_output=True, text=True,
    )
    _BUILD_LOG["stdout"], _BUILD_LOG["stderr"] = proc.stdout, proc.stderr
    if proc.returncode != 0:
        raise RuntimeError(
            "lesson.cpp did not compile. The compiler's FIRST complaint is usually the real "
            f"one, and it names a line:\n{proc.stdout}\n{proc.stderr}"
        )
    _BUILD_LOG["built"] = True
    return CPP_BIN


def run_cpp(args: list) -> dict:
    """Run the binary and parse its `tag value...` output into a dict.

    Exit code 3 means an exercise still has its TODO, and becomes a Python
    NotImplementedError, so the grader can tell "not written yet" from "written and wrong".
    """
    binary = build_cpp()
    proc = subprocess.run([str(binary), *args], capture_output=True, text=True, cwd=LESSON_ROOT)
    if proc.returncode == 3:
        raise NotImplementedError(proc.stderr.strip() or "an exercise still has its TODO")
    if proc.returncode != 0:
        raise RuntimeError(f"{binary.name} {' '.join(args[:2])} exited {proc.returncode}:\n"
                           f"{proc.stdout}\n{proc.stderr}")
    out = {"pair_counts": {}, "merges": [], "raw": proc.stdout}
    for line in proc.stdout.splitlines():
        parts = line.split()
        if not parts:
            continue
        tag, rest = parts[0], parts[1:]
        if tag == "pair":
            out["pair_counts"][(int(rest[0]), int(rest[1]))] = int(rest[2])
        elif tag == "merge":
            out["merges"].append(tuple(int(x) for x in rest[1:5]))
        elif tag == "best":
            out["best"] = None if rest[0] == "none" else tuple(int(x) for x in rest[:3])
        elif tag == "seq":
            out["seq"] = [int(x) for x in rest[0].split(",")] if rest else []
        elif tag == "train_seconds":
            out["train_seconds"] = float(rest[0])
        elif tag in ("pairs", "length", "bytes", "symbols", "final_symbols"):
            out[tag] = int(rest[0])
    return out


def cpp_selftest() -> str:
    """`make test` — your binary checking its own four functions. Returns its output."""
    build_cpp()
    proc = subprocess.run(
        ["make", "-C", str(LESSON_ROOT), f"SRC={CPP_SRC.relative_to(LESSON_ROOT)}",
         f"BIN={CPP_BIN.relative_to(LESSON_ROOT)}", "test"],
        capture_output=True, text=True,
    )
    if "NOT_IMPLEMENTED" in proc.stderr:
        # The binary's own line names the function; make's "Error 3" line after it does not.
        raise NotImplementedError(next(line for line in proc.stderr.splitlines()
                                       if "NOT_IMPLEMENTED" in line))
    if proc.returncode != 0:
        raise AssertionError(f"make test failed:\n{proc.stdout}\n{proc.stderr}")
    return proc.stdout


def cpp_count_pairs(seq: list) -> dict:
    """Your count_pairs, as a Python dict."""
    return run_cpp(["count", "--seq", ",".join(str(s) for s in seq)])["pair_counts"]


def cpp_best_pair(counts: dict):
    """Your best_pair, handed counts directly — it does not depend on exercise 1."""
    packed = ",".join(f"{a}:{b}:{n}" for (a, b), n in counts.items())
    return run_cpp(["best", "--counts", packed])["best"]


def cpp_apply_merge(seq: list, a: int, b: int, new_id: int) -> list:
    """Your apply_merge, as a Python list."""
    return run_cpp(["merge", "--seq", ",".join(str(s) for s in seq),
                    "--a", str(a), "--b", str(b), "--new-id", str(new_id)])["seq"]


def cpp_train(corpus_path=None, merges: int = HEADLINE_MERGES) -> dict:
    """Your train, on a corpus file, including the binary's own timing of the loop."""
    return run_cpp(["train", "--corpus", str(corpus_path or CORPUS), "--merges", str(merges)])


def oracle_corpus(nbytes: int = ORACLE_BYTES) -> Path:
    """Write a fixed slice of the corpus to build/, so both trainers read the same file."""
    blob = CORPUS.read_bytes()[:nbytes]
    while blob and not blob[-1:].isspace():
        blob = blob[:-1]
    path = LESSON_ROOT / "build" / f"oracle_{CPP_BIN.name}.txt"
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(blob)
    return path


if __name__ == "__main__":
    _try("make test", lambda: print(cpp_selftest()))

## 4. Exercise 1 — `count_pairs`

Open `lesson.cpp`, find EXERCISE 1, fill it in. One scan, one local holding the previous
symbol, and no pair counted when either side is `kBoundary`. Then run this check.

<details><summary>💡 Hint 1 — what to think about</summary>

A pair has two sides, and either one can be the word boundary: check both before you count.
Then ask what your loop does with an empty sequence, a one-symbol sequence, and the very
last pair. Counting is allowed to overlap here — only merging is not — so do not skip ahead
after a hit.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Start at the second element and walk to the end once, keeping the previous symbol in a local
`int`. At each position, if neither the previous nor the current symbol is `kBoundary`, add
one to the entry for their packed key — increment, never assign. Then the current symbol
becomes the previous one. Starting at index 1 means a sequence shorter than two symbols never
enters the loop at all. Keep every symbol an `int`: after the first merge, ids pass 255.

</details>

In [ ]:
def _check_count_pairs() -> None:
    build_cpp(force=True)
    seq = pretokenise(b"ab ab")
    got, want = cpp_count_pairs(seq), count_pairs_py(seq)
    assert got == want, (
        f"count_pairs on 'ab ab' gave {got}, expected {want} — entries mentioning {BOUNDARY} "
        "mean you are counting across the word boundary; every count equal to 1 means you are "
        "assigning where you meant to increment."
    )
    overlap = cpp_count_pairs([7, 7, 7, BOUNDARY])
    assert overlap == {(7, 7): 2}, (
        f"[7,7,7] should report (7,7) twice, got {overlap} — counting is allowed to overlap; "
        "only merging is not."
    )
    assert cpp_count_pairs([]) == {} and cpp_count_pairs([5]) == {}, (
        "a sequence shorter than two symbols has no pairs — guard the loop rather than "
        "indexing seq[i+1] past the end."
    )
    big = corpus_symbols(limit=20_000)
    mine = cpp_count_pairs(big)
    assert mine == count_pairs_py(big), (
        "your counts match on toy input but not on real text — the usual causes are holding "
        "symbols in a char (they exceed 255 after the first merge) and dropping the last pair."
    )
    print(f"exercise 1 looks right: {len(mine)} distinct pairs across {len(big)} real symbols")


if __name__ == "__main__":
    _try("exercise 1", _check_count_pairs)

## 5. Exercise 2 — `best_pair`, and why ties matter

Two pairs tie at the top. Whichever you return becomes token 257, which changes every merge
after it. So the tie-break is not a detail: it is the difference between a vocabulary you can
ship and one that differs between runs. `std::unordered_map` promises you no order at all.

<details><summary>💡 Hint 1 — what to think about</summary>

Two questions pick the winner: which count is highest, and, when counts are equal, which
pair is smaller. Iteration order must never get a vote — a `best_pair` that keeps "the first
maximum I met" passes a small case by luck and fails the moment the same pairs arrive in
another order. And decide what an empty map does to the three out-parameters: nothing.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Keep a leader — its packed key and its count — and a flag saying whether you have one yet.
For each entry, take it as the new leader if its count is strictly greater, or if the counts
are equal and its packed key is smaller. The packed key orders by `a` first, then `b`, which
is exactly the tie-break. Only after the loop, and only if you found a leader, unpack it,
write through the three pointers and return `true`. An empty map returns `false`.

</details>

In [ ]:
def _check_best_pair() -> None:
    build_cpp()
    tie = cpp_best_pair({(9, 9): 4, (2, 7): 4})
    assert tie == (2, 7, 4), (
        f"a tie between (9,9) and (2,7) must go to (2,7), got {tie} — comparing the packed "
        "keys does this for you; iteration order does not."
    )
    assert cpp_best_pair({(1, 2): 3, (4, 5): 9, (6, 7): 5}) == (4, 5, 9), (
        "the highest count must win outright when nothing ties — check the comparison is > "
        "and not >=, which lets a later equal-count pair displace the leader."
    )
    assert cpp_best_pair({}) is None, (
        "an empty count map must return false, not a pair of zeros — train relies on that "
        "signal to stop."
    )
    # The check the toy cases above cannot make. One tied set, three insertion orders: a real
    # tie-break answers the same every time, while "the first maximum I met" follows whatever
    # order the hash table happened to produce.
    _tied = [(3, 4), (17, 18), (41, 42), (88, 89), (150, 151), (260, 261), (301, 302)]
    answers = {cpp_best_pair({pair: 9 for pair in order})
               for order in (_tied, list(reversed(_tied)), _tied[3:] + _tied[:3])}
    assert answers == {(3, 4, 9)}, (
        f"the same seven tied pairs produced {sorted(answers)} — the only thing that changed "
        "was the order they were inserted in, so you are returning the first maximum the "
        "container handed you rather than the smallest (a,b). This is the check the small "
        "cases above can pass without a tie-break at all; compare the packed keys."
    )
    counts = count_pairs_py(corpus_symbols(limit=20_000))
    mine, reference = cpp_best_pair(counts), best_pair_py(counts)
    assert mine == reference, (
        f"on real text your winner is {mine} and Python's is {reference} — same tie-break, or "
        "the two merge lists diverge at the first tie."
    )
    print(f"exercise 2 looks right: winner on a real slice is {mine}")


if __name__ == "__main__":
    _try("exercise 2", _check_best_pair)

## 6. Exercise 3 — `apply_merge`, in place

Two cursors over one vector: read and write. The write cursor can never overtake the read
cursor, so there is no need for a second array — and no allocation inside the hot loop.

<details><summary>💡 Hint 1 — what to think about</summary>

After a match, both symbols are used up: how far must the read cursor move so neither can
take part in another match? Before you look at a symbol's right-hand neighbour, is there one?
And the function hands back two answers — the vector's new size and the returned length —
which must agree. Both come from the same cursor, and it is not the read cursor.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Start a read cursor and a write cursor at zero. While the read cursor is inside the vector:
if it has a right-hand neighbour and the two symbols there are `a` then `b`, store `new_id`
at the write cursor and move the read cursor past both; otherwise copy the one symbol across
and move the read cursor past it. Either way the write cursor moves by one. When the scan
ends, `resize()` to the write cursor and return that same value.

</details>

In [ ]:
def _check_apply_merge() -> None:
    build_cpp()
    triple = cpp_apply_merge([7, 7, 7], 7, 7, 300)
    assert triple == [300, 7], (
        f"[7,7,7] merged on (7,7) must be [300,7], got {triple} — after a match, advance the "
        "read cursor by two; advancing by one consumes the same symbol twice."
    )
    assert cpp_apply_merge([1, 2, 3], 8, 9, 300) == [1, 2, 3], (
        "a pair that does not occur must leave the sequence untouched — your resize() is "
        "probably using the read cursor instead of the write cursor."
    )
    assert cpp_apply_merge([1, 2, 1, 2], 1, 2, 300) == [300, 300], (
        "two separate occurrences must both merge — do not stop at the first match."
    )
    assert cpp_apply_merge([5, 1, 2], 1, 2, 300) == [5, 300], (
        "symbols before a match must be copied through, in order."
    )
    # The other half of the contract: the LENGTH your function returns. It is easy to rewrite
    # the sequence perfectly and still hand back the wrong number.
    reported = run_cpp(["merge", "--seq", "7,7,7,7,7", "--a", "7", "--b", "7", "--new-id", "300"])
    assert reported["length"] == len(reported["seq"]), (
        f"apply_merge rewrote [7,7,7,7,7] to {reported['seq']} — which is right — but returned "
        f"a length of {reported['length']} instead of {len(reported['seq'])}. Return the WRITE "
        "cursor. The read cursor, and seq.size() taken before the resize(), are both the "
        "length you started with."
    )
    seq = corpus_symbols(limit=20_000)
    a, b, _ = best_pair_py(count_pairs_py(seq))
    mine = cpp_apply_merge(seq, a, b, 999)
    assert mine == apply_merge_py(seq, a, b, 999), (
        "toy cases pass but real text does not — the classic cause is a match at the very "
        "last position, where seq[i+1] is off the end."
    )
    print(f"exercise 3 looks right: merging ({a},{b}) takes {len(seq)} symbols to {len(mine)}")


if __name__ == "__main__":
    _try("exercise 3", _check_apply_merge)

## 7. Exercise 4 — `train`, and the proof that it matches Python

Now the loop: count, pick, record, apply, repeat. Stop when the best pair occurs fewer than
`kMinCount` times. The test that matters is not "does this look plausible" — it is whether
your merge list is **identical** to the Python trainer's on the same bytes.

<details><summary>💡 Hint 1 — what to think about</summary>

Three decisions make your list match Python's merge for merge: when you stop, what id each
merge gets, and whether each merge is applied before the next count. Read the stop rule's
boundary twice — a pair seen *exactly* `kMinCount` times is still worth merging. And a merge
you record but never apply shows up as the right list with the wrong final length.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Loop `k` from zero while `k` is below `n_merges`. Each pass: recount the whole sequence with
exercise 1, ask exercise 2 for the winner, and break — before recording anything — if there
is no winner or its count is strictly below `kMinCount`. Otherwise record a `Merge` whose id
is `kFirstMergeId + k`, apply it to `seq` with exercise 3, and go round again. Return the
merges in the order you made them.

</details>

In [ ]:
def _check_train() -> None:
    build_cpp()
    path = oracle_corpus()
    cpp = cpp_train(path, ORACLE_MERGES)
    ref = train_py(pretokenise(path.read_bytes()), ORACLE_MERGES)
    assert len(cpp["merges"]) == len(ref["merges"]), (
        f"C++ made {len(cpp['merges'])} merges, Python {len(ref['merges'])} — too few means "
        "your stop rule fires early (it is count < kMinCount, tested before recording); too "
        "many means you are not stopping at n_merges."
    )
    for k, (got, want) in enumerate(zip(cpp["merges"], ref["merges"])):
        assert got == want, (
            f"merge {k} differs: C++ {got}, Python {want}. Everything before it agrees, so the "
            f"divergence starts here. A wrong new_id means you are not numbering from "
            f"{FIRST_MERGE_ID}+k; a different pair at equal counts is the tie-break."
        )
    assert cpp["final_symbols"] == ref["final_symbols"], (
        f"identical merges but different final lengths ({cpp['final_symbols']} against "
        f"{ref['final_symbols']}) — train is recording merges it never applies."
    )
    # The boundary of the stop rule, which a corpus this size never exercises: every pair in
    # "ab ab cd cd" occurs exactly MIN_COUNT times, so both merges are worth making.
    _edge_path = LESSON_ROOT / "build" / f"stoprule_{CPP_BIN.name}.txt"
    _edge_path.parent.mkdir(parents=True, exist_ok=True)
    _edge_path.write_bytes(b"ab ab cd cd")
    edge = cpp_train(_edge_path, 5)["merges"]
    assert edge == [(97, 98, 257, 2), (99, 100, 258, 2)], (
        f"in 'ab ab cd cd' every pair occurs exactly MIN_COUNT ({MIN_COUNT}) times, so both "
        f"merges should be made; got {edge}. An empty list means your stop rule is "
        "`count <= kMinCount` where the spec says `count < kMinCount` — a pair seen exactly "
        "kMinCount times still buys compression."
    )
    print(f"exercise 4 looks right: {len(cpp['merges'])} merges identical to Python, "
          f"{cpp['symbols']} symbols down to {cpp['final_symbols']}")


if __name__ == "__main__":
    _try("exercise 4", _check_train)

## 8. The measurement

Same corpus, same algorithm, same merge list, two languages. The only honest comparison
times the training loop alone — not file reading, not pre-tokenisation, and certainly not
compilation. Every number printed below comes from the run you just did.

In [ ]:
def speed_report(merges: int = HEADLINE_MERGES) -> dict:
    """Train both implementations on the full corpus and return the measured comparison."""
    cpp = cpp_train(CORPUS, merges)
    ref = train_py(corpus_symbols(), merges)
    return {
        "bytes": CORPUS.stat().st_size,
        "symbols": cpp["symbols"],
        "merges": len(cpp["merges"]),
        "python_seconds": ref["seconds"],
        "cpp_seconds": cpp["train_seconds"],
        "speedup": ref["seconds"] / cpp["train_seconds"],
        "identical_merges": list(cpp["merges"]) == list(ref["merges"]),
        "final_symbols": cpp["final_symbols"],
    }


def _check_speedup() -> None:
    report = speed_report()
    assert report["identical_merges"], (
        "the trainers disagree on the full corpus even though the slice matched — a speed "
        "comparison between two different answers is worthless; fix exercise 4 first."
    )
    assert report["speedup"] > 1.5, (
        f"C++ came out only {report['speedup']:.2f}x faster — check that you built with -O2 "
        "(the Makefile does), that apply_merge does not allocate a fresh vector per merge, "
        "and that nothing prints inside the loop."
    )
    print(f"corpus        {report['bytes']} bytes -> {report['symbols']} symbols")
    print(f"merges        {report['merges']} (identical lists: {report['identical_merges']})")
    print(f"compression   {report['symbols']} -> {report['final_symbols']} symbols")
    print(f"python        {report['python_seconds']:.3f} s")
    print(f"c++           {report['cpp_seconds']:.3f} s")
    print(f"speedup       {report['speedup']:.1f}x on this machine")


if __name__ == "__main__":
    _try("the measurement", _check_speedup, needs=("exercise 4",))

## 9. What the speedup is, and what it is not

Your C++ is faster because every symbol is an `int` in a flat array, every pair is one
64-bit key, and the loop compiles to instructions instead of bytecode dispatch. That is a
**constant factor**. Both programs still do `n_merges × n_symbols` work, so doubling the
corpus still doubles both. Run this and watch the C++ curve keep Python's shape.

In [ ]:
def cpp_scaling(fractions=(0.25, 0.5, 1.0), merges: int = 30) -> list:
    """Time the C++ trainer on the same slices section 2 used, for the same merge count."""
    total = CORPUS.stat().st_size
    rows = []
    for fraction in fractions:
        result = cpp_train(oracle_corpus(int(total * fraction)), merges)
        rows.append({"symbols": result["symbols"], "seconds": result["train_seconds"],
                     "ns_per_symbol_per_merge": 1e9 * result["train_seconds"]
                     / (result["symbols"] * merges)})
    return rows


def _show_scaling() -> None:
    """Both scaling tables side by side. C++ runs first, so an unfinished `train` skips at
    once instead of after Python has spent its seconds."""
    cpp_rows = cpp_scaling()
    py_rows = python_scaling()
    print("per symbol, per merge — the only column that can tell a constant from a curve")
    print(f"{'symbols':>9} {'python':>11} {'c++':>11} {'ratio':>9}")
    for p, c in zip(py_rows, cpp_rows):
        print(f"{c['symbols']:9d} {p['ns_per_symbol_per_merge']:8.1f} ns"
              f" {c['ns_per_symbol_per_merge']:8.1f} ns"
              f" {p['ns_per_symbol_per_merge'] / c['ns_per_symbol_per_merge']:8.1f}x")
    print("\nRead the two middle columns DOWN, not across. Each is roughly flat as the corpus")
    print("grows, and that flatness is the straight line — the same line in both languages.")
    print("The ratio column is the whole of what the language bought: a constant. To bend the")
    print("line you would have to stop rescanning — keep the counts between merges and update")
    print("only the positions the merge touched. That is an algorithmic change, and no amount")
    print("of C++ substitutes for it.")


if __name__ == "__main__":
    _try("the scaling table", _show_scaling, needs=("exercise 4",))

## 10. The honest footnote

The argument here is *systems language*, not *C++ specifically* — and the evidence cuts both
ways, so here it is straight (every source in `claims.yaml`):

- **HuggingFace `tokenizers`, the trainer behind most published BPE vocabularies, is Rust**,
  with bindings for Python, Node and Ruby. **OpenAI's `tiktoken` is Rust too**, behind PyO3.
- **Google's SentencePiece is C++**, and much of the research world trains with it.

Rust wins those choices on memory safety and tooling, not on the loop you just wrote: a hash
lookup and a two-cursor scan compile to the same shape in either. What does not change is the
thing you measured — the interpreter was the cost, not the syntax.

`tokenizers` is installed here, so run the real thing on your corpus. It is deliberately
*not* a like-for-like race: it counts over a word-frequency table instead of rescanning a
flat array, which is exactly the algorithmic fix section 9 described. You are seeing both
wins at once, which is why the gap is bigger than yours.

In [ ]:
def production_reference(vocab_size: int = 256 + HEADLINE_MERGES + 1) -> dict:
    """Train HuggingFace's Rust BPE on the same corpus. A different algorithm, on purpose.

    The vocabulary budget is derived rather than typed: one slot per byte value, one per merge
    this lesson asks for, plus the unknown token, so both trainers are at least asked for the
    same size. It is still not a like-for-like race, for the reason in the text above.
    """
    from tokenizers import Tokenizer, models, pre_tokenizers, trainers
    tokenizer = Tokenizer(models.BPE())
    tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
    trainer = trainers.BpeTrainer(vocab_size=vocab_size, show_progress=False)
    words = CORPUS.read_text(encoding="utf-8", errors="replace").split()
    start = time.perf_counter()
    tokenizer.train_from_iterator(words, trainer)
    return {"seconds": time.perf_counter() - start, "vocab": tokenizer.get_vocab_size()}


if __name__ == "__main__":
    try:
        import tokenizers
        _hf = production_reference()
        print(f"huggingface tokenizers {tokenizers.__version__} (Rust): {_hf['seconds']:.3f} s "
              f"for a vocabulary of {_hf['vocab']}")
        print("different algorithm, different tie-breaks, different vocabulary — read that")
        print("number as scale, not as a scoreboard.")
    except Exception as _exc:                      # noqa: BLE001 - an optional demonstration
        print("skipping the production reference:", _exc)

## 11. Common mistakes

- **Counting pairs across the boundary.** You invent tokens spanning two words and diverge
  from Python at the first merge. Skip the pair; do not record it as zero.
- **Advancing by one after a match in `apply_merge`.** `[7,7,7]` becomes `[300,300,7]`: the
  middle 7 is consumed twice, the sequence barely shrinks, and the final length disagrees
  with Python's. Do not take that on trust either — the cell below runs the bug.
- **Letting hash order pick the winner.** It passes today and fails on another machine, or
  after the corpus grows by one word. The tie-break *is* the reproducibility guarantee.
- **`char` or `uint8_t` for symbols.** The 257th symbol has nowhere to live. Use `int`.
- **Building a fresh `std::vector` per merge.** Correct, but it allocates and copies hundreds
  of times, and hands back much of the constant factor you came for.
- **`std::map` instead of `std::unordered_map`.** A tree costs a `log n` factor of cache
  misses per probe to buy an ordering you need exactly once — at the tie-break, which the
  packed key already gives you for free.
- **Timing the wrong thing.** Including `make`, file reading or pre-tokenisation measures the
  harness. The binary times the loop and nothing else; match that in anything you report.
- **Benchmarking a `-O0` build.** Unoptimised C++ can lose to Python outright. The Makefile
  uses `-O2`; if you compile by hand, match it.
- **Wrapping a function body in `catch (...)` and returning a default.** An empty merge list
  is a plausible-looking answer, so the grader reports a wrong answer where it would have
  reported "not written yet" — and the `NOT_IMPLEMENTED` signal the notebook relies on to
  tell those two apart never reaches it. Let the exception out.
- **A binary left over from before you moved or rebuilt your checkout.** `make` decides what
  is stale by comparing timestamps, and a binary built yesterday in the old location is still
  newer than its source — so `make` reports "up to date" and hands you the old executable.
  On the compiled lessons that link a shared library (`libmujoco`, in the humanoid track) the
  symptom is louder, because the linker bakes the library's ABSOLUTE path into the binary:
  move the checkout, or rebuild the virtualenv somewhere else, and the binary dies at startup
  with `dyld: Library not loaded`, naming a path that no longer exists. Rebuilding does not
  help; `make` still thinks there is nothing to do. **The fix is `make clean`**, then build
  again — run it as a reflex after moving the checkout or rebuilding the environment.

In [ ]:
def apply_merge_advancing_by_one(seq: list, a: int, b: int, new_id: int) -> list:
    """`apply_merge_py` with one character changed: `i += 2` became `i += 1`.

    This is the commonest wrong answer to exercise 3. It is here so you can watch what it
    does rather than guess — and so the numbers in the bullet above are printed, not typed.

    Example:
        >>> apply_merge_advancing_by_one([7, 7, 7], 7, 7, 300)
        [300, 300, 7]
    """
    out, i, n = [], 0, len(seq)
    while i < n:
        if i + 1 < n and seq[i] == a and seq[i + 1] == b:
            out.append(new_id)
            i += 1                      # the bug: the second symbol is read a second time
        else:
            out.append(seq[i])
            i += 1
    return out


if __name__ == "__main__":
    print(f"{'input':>18}  {'correct':<22} advance-by-one")
    for _case in ([7, 7, 7], [7, 7, 7, 7], [7, 7, 7, 7, 7]):
        print(f"{str(_case):>18}  {str(apply_merge_py(_case, 7, 7, 300)):<22}"
              f" {apply_merge_advancing_by_one(_case, 7, 7, 300)}")
    print("\nThe surplus symbols on the right are the ones consumed twice. A run of n sevens")
    print("should collapse to about n/2 symbols; the bug leaves it barely shorter than n, so")
    print("the sequence stops shrinking and your final length parts company with Python's.")

## 12. Self-check

1. Your `best_pair` returns the first maximum it meets while iterating the
   `std::unordered_map`, and every test passes on your machine. What is actually wrong?
   - (a) nothing — the standard fixes iteration order
   - (b) it is slower than comparing packed keys
   - (c) two runs, two standard-library versions or two corpus sizes can produce different
         vocabularies from identical input, and the vocabulary is the artefact you ship
   - (d) it only breaks when two pairs both have count zero

2. `apply_merge` on `[7, 7, 7, 7]`, merging `(7,7)` into `300`, must give:
   - (a) `[300, 300]`  - (b) `[300, 7, 7]`  - (c) `[300, 300, 300, 7]`  - (d) `[300, 7]`

3. Your C++ beat Python by the factor you measured. Where did that come from?
   - (a) a better algorithm — C++ avoids the rescan
   - (b) the same `n_merges × n_symbols` work at a much lower cost per symbol: no boxed
         integers, no tuple keys, no interpreter dispatch
   - (c) multiple cores
   - (d) the compiler recognises BPE and replaces it

4. Which is true of the library most published BPE vocabularies are trained with?
   - (a) it is C++, which is why this lesson is in C++
   - (b) it is pure Python, and slow
   - (c) it is Rust, and this lesson's argument is about systems languages rather than C++
   - (d) it is C++, but only in the training path

5. You double the corpus and halve the merge count. Roughly what happens to both runtimes?
   - (a) both unchanged — the product is unchanged  - (b) both double  - (c) both halve
   - (d) Python doubles, C++ is unchanged

Answers, with reasoning, are in this lesson's worked solution in the course repository.

## What you built, and where it goes next

A tokenizer trainer in C++ whose merge list is provably identical to the Python one, a
measurement of what the language was worth on your own machine, and the sharper lesson
underneath it: the language bought a constant, and the algorithm is still the naive rescan.
F02 picks this up where the vocabulary becomes an encoder and the rescan becomes incremental.
The cell below re-runs the four correctness checks and prints your progress board.

In [ ]:
# Your progress board. The exercises it lists are the ones `_EXERCISES` names in the setup cell.
_MARKS = {"passed": "✅ passed     ", "failed": "❌ failed     ", "not started": "⏳ not started"}


def _progress_board() -> None:
    """One line per exercise, from what the checks recorded — then how many are complete."""
    print("\nyour progress")
    done, next_up = 0, None
    for label, function, where in _EXERCISES:
        status = _STATUS.get(label, "not started")
        done += status == "passed"
        waiting = _WAITING_ON.get(label)
        note = (f"   (waiting on `{waiting}`)"
                if status == "not started" and waiting not in (None, "", function) else "")
        print(f"  {_MARKS[status]}  {label} · {function}{note}")
        if status != "passed" and next_up is None:
            next_up = (label, function, where)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    if next_up:
        print(f"next: {next_up[0]} — `{next_up[1]}` in lesson.cpp, {next_up[2]}. Its cell has "
              "hints if you are stuck.")


if __name__ == "__main__":
    # A final sweep of the correctness checks. The measurement in section 8 is deliberately
    # not repeated here: it is the slowest cell in the lesson and nothing below it changed.
    for _label, _check in (("exercise 1", _check_count_pairs), ("exercise 2", _check_best_pair),
                           ("exercise 3", _check_apply_merge), ("exercise 4", _check_train)):
        _try(_label, _check)
    _progress_board()
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends the run non-zero, so a green exit code can never
    # paper over it. In a notebook kernel the same news is a printed line, not a traceback.
    _failed = [_label for _label, _status in _STATUS.items() if _status == "failed"]
    if _failed and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(_failed))
    if _failed:
        print("checks failed: " + ", ".join(_failed) + " — each one printed its likely mistake "
              "where it ran.")